<a href="https://colab.research.google.com/github/kimdonggyu2008/Personal_Study/blob/main/%EC%9D%8C%EC%84%B1%EC%9D%B8%EC%8B%9D_%EB%AA%A8%EB%8D%B8%ED%8C%8C%ED%8A%B8_%EA%B5%AC%ED%98%84%EC%97%B0%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ctc

In [ ]:
import math
import numpy as np

LOG_ZERO = -float("inf")
LOG_ONE = 0.0

# 문자 인덱스 → 문자 매핑 예시
classes = ['a', 'b', 'c', '_']  # 마지막은 blank
blank_index = len(classes) - 1


class BeamEntry:
    def __init__(self):
        self.y = ()                  # 라벨 시퀀스 (튜플)
        self.prBlank = LOG_ZERO     # blank로 끝나는 확률 (log scale)
        self.prNonBlank = LOG_ZERO  # 문자로 끝나는 확률 (log scale)
        self.prTotal = LOG_ZERO     # 전체 확률 (log scale)


In [ ]:

class BeamState:
  def __init__(self):
    self.entries={}

  def sort(self):
    return sorted(self.entries.values(), key=lambda x :self.entries[x].prTotal,reverse=True)


  def norm(self):
    for i in self.entries:
      l=len(i)
      if l>0:
        self.entries[i].PrTotal/=1


  def log_add_prob(log_x,log_y):
    if log_x<=LOG_ZERO:
      return log_y
    if log_y<=LOG_ZERO:
      return log_x
    return log_x+math.log(1+math.exp(log_y-log_x))

  def calc_ext_pr(self,k,y,t,mat,beam_state):
    #k는 인덱스, y는 시퀀스, t는 타임스텝,mat은 확률행렬, beam_state는 이전 t의 beam_state
    bigram_log_prob=0.0

    if self.lm:
      c1=self.classes[y[-1]] if len(y)>0 else ""
      c2=self.classes[k]
      lm_prob=self.lm.get_bi_prob(c1,c2)
      bigram_log_prob=math.log(lm_prob)

    log_prob_k=math.log(mat[t,k]) #행렬과 곱해서 상관관계 추출

    if len(y)>0 and y[-1]==k and mat[t-1,self.blank_index]<0.9: #끝일 확률이 낮을때
      return log_prob_k + bigram_log_prob + beam_state.entries[y].prBlank
    else:
      return log_prob_k + bigram_log_prob + beam_state.entries[y].prTotal


  def add_labelling(self,beam_state,y):
    if y not in beam_state.entries:
      beam_state.entries[y]=BeamEntry()

  def decode(self,inputs,input_lengths):
    B,T,C=inputs.shape
    results=[]

    for b in range(B):
      mat=inputs[b].numpy()
      beam=BeamState()

      y0=()
      beam_entries[y0]=BeamEntry()
      beam_entries[y0].prBlank=LOG_ONE
      beam_entries[y0].prTotal=LOG_ONE

      for t in range(T):
        next_beam=BeamState()

        if (1-mat[t,self.blank_index])<0.1:
          continue

        top_k=beam.sort()[0:self.beam_width]

        for y in top_k:
          entry=beam.entries[y]

          pr_blank=entry.PrTotal+math.log(1-mat[t,self.blank_index]) #빈칸이 될 확률은 조금씩 올라감
          pr_non_blank=LOG_ZERO #빈칸이 아닐확률은 계속 작음
          if len(y)>0:
            pr_non_blank=self.calc_ext_pr(k,y,t,mat,beam) #길이가 길어지면 조금씩 변함

          self.add_labelling(next_beam,y)
          be=next_beam.entries[y]
          be.y=y
          be.prBlank=self.log_add_prob(be.prBlank,pr_blank)
          be.prNonBlank=self.log_add_prob(be.prNonBlank,pr_non_blank)
          be.prTotal=log_add_prob(be.prBlank,be.prNonBlank)


          for k in range(C):
            if k==self.blank_index:
              continue

            new_y=y+(k,)
            pr_k=self.calc_ext_pr(k,y,t,mat,beam)
            self.add_labelling(next_beam,new_y)
            be_k=next_beam.entries[new_y]
            be_k.y=new_y
            be_k.prNonBlank=self.log_add_prob(be_k.prNonBlank,pr_k)
            be_k.prBlank=self.log_add_prob(be_k.prTotal,pr_k)

        beam=next_beam

      top_k=beam.sort()[0:self.beam_width]
      final_beam=BeamState()
      for y in k:
        c1=self.classes[y[-1]] if len(y)>0 else ""
        c2=""
        lm_score=0
        if self.lm:
          lm_score=math.log(self.lm.get_bi_prob(c1,c2))*self.lm_alpha
        pr=beam.entries[y].prTotal+lm_score

        self.add_labelling(final_beam,y)
        fe=final_beam.entries[y]
        fe.y=y
        fe.prTotal=log_add_prob(fe.prTotal,pr)

      final_beam.norm()
      best=final_beam.sort()[0]

      result=''.join([self.classes[i] for i in best])
      results.append(result)

    return results

RNN-T

              음성 입력 (x) ───────▶ Encoder ──────┐
                                                 ▼
                      이전 출력 시퀀스 (y) ─▶ Prediction Network
                                                 ▼
                                ┌─────── Joint Network ───────┐
                                │                            ▼
                        확률 분포 logit[t,u,v]       (t: 인코더, u: 디코더)



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

BLANK_IDX=0

In [ ]:
class Encoder(nn.Module):
  def __init__(self.input_dim,hidden_dim):
    super.__init__()
    self.lstm=nn.LSTM(input_dim,hidden_dim,batch_first=True,bidirectiona=True)

  def forward(self,x):
    out,_=self.lstm(x)
    return out

In [ ]:
class PredictionNet(nn.Module):
  def __init__(self,vocab_size,hidden_dim):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,hidden_dim)
    self.lstm=nn.LSTM(hidden_dim,hidden_dim,batch_first=True)

  def forward(self,y):
    emb,_=self.embedding(y)
    x=lstm(emb)
    return x

In [ ]:
class JointNet(nn.Module):
  def __init__(self,hidden_dim,vocab_size):
    super().__init__()
    self.hidden_dim=hidden_dim
    self.vocab_size=vocab_size

  def forward(self,h_enc,h_pred):
    T,H=h_enc.shape
    U,_=h_pred.shape
    h_enc=h_enc.unsqueeze(1).expand(T,U,H)
    h_pred=h_pred.unsqueeze(0).expand(T,U,H)

    concat=torch.cat([h_enc,h_pred],dim=-1)
    out=self.ffn(concat)
    return out


In [ ]:
class RNNTModel(nn.Module):
  def __init__(self,input_dim,hidden_dim,vocab_size):
    super().__init__()
    self.encoder=Encoder(input_dim,hidden_dim)
    self.pred_net=PredictionNet(vocab_size,hidden_dim)
    self.joint_net=JointNet(hidden_dim,vocab_size)

  def forward(self,x,y):
    h_enc=self.encoder(x)[0]
    h_pred=self.pred_net(y)[0]

    logits=[]
    for b in range(x.size(0)):
      out=self.joint_net(h_enc[b],h_pred[b])
      logits.append(out)
    return torch.stack(logits,dim=0)

In [ ]:
class greedy_decode(model,x,max_u=50):
  model.eval()
  with torch.no_grad():
    h_enc=model.encoder(x)[0][0]
    pred=torch.tensor([[BLANK_IDX]],dtype=torch.long)
    h_pred=model.pred_net(pred)[0][0]

    y=[]
    t,u=0,0

    while t<h_enc.size(0) and u<max_u:
      logit=model.joint_net(h_enc[t],h_pred[u])
      prob=F.log_softmax(logit,dim=-1)
      v=torch.argmax(prob,dim=-1)

      if v==BLANK_IDX:
        t+=1
      else:
        y.append(v)
        u+=1
        new_input=torch.tensor([[v]],dtype=torch.long)
        h_pred=model.pred_net(new_input)[0][0]
    return y


#LAS


음성 입력 x ─▶ Encoder ──▶ Context vector ─┬────► Decoder ──▶ 예측된 y 시퀀스

                                      ▲         │
                                      └────Attention────┘

In [ ]:
class Encoder(nn.Module):
  def __init__(self,input_dim,hidden_dim):
    super().__init__()
    self.lstm=nn.LSTM(input_dim,hidden_dim,batch_first=True,bidirectional=True)
    self.linear=nn.Linear(hidden_dim*2,hidden_dim)

  def forward(self,x):
    h,_=self.lstm(x)
    h=self.linear(h)
    return h

In [ ]:
class DotAttention(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self,encoder_outputs,decoder_hidden):
    scores=torch.bmm(encoder_outputs,decoder_hidden.unsqueeze(2)).squeeze(2)
    attn_weights=F.softmax(scores,dim=-1)
    context=torch.bmm(attn_weights.unsqueeze(1),encoder_outputs).squeeze(1)
    return context,attn_weights


In [ ]:
class Decoder(nn.Module):
  def __init__(self,vocab_size,hidden_dim):
    self.lstm=nn.LSTMCell(hidden_dim+hidden_dim,hidden_dim)
    self.out=nn.Linear(hidden_dim,vocab_size)
    self.attention=DotAttention()

  def forward(self,encoder_outputs,decoder_hidden):
    B,T,H=encoder_outputs.size()
    U=target.size(1)

    outputs=[]
    h,c=encoder_outputs.new_zeros(B,H),encoder_outputs.new_zeros(B,H)
    input_token=targets[:,0]

    for u in range(1,U):
      embed=self.embedding(input_token)
      context,attn=self.attention(encoder_outputs,h)
      lstm_input=torch.cat([embed,context],dim=-1)
      h,c=self.lstm(lstm_input,(h,c))
      logit=self.out(h)
      outputs.append(logit.unsqueeze(1))

      teacher_force=torch.rand(1).item()<teacher_forcing_ratio
      top1=logits.argmax(dim=-1)
      input_token=targets[:,u] if teacher_force else top1

  return torch.cat(outputs,dim=1)

In [ ]:
class LAS(nn.Module):
  def __init__(self,input_dim,hidden_dim,vocab_size):
    super().__init__()
    self.encoder=Encoder(input_dim,hidden_dim)
    self.decoder=Decoder(vocab_size,hidden_dim)

  def forward(self,x,y):
    enc_out=self.encoder(x)
    out=self.decoder(enc_out,y)
    return out

# 하이브리드 모델


입력 x ───▶ Encoder ──┬────────▶ CTC Loss

                     │

                     └───────▶ Attention Decoder ──▶ Cross Entropy Loss
